# Reinforcement learning basics: REINFORCE on a grid world

Every other notebook in this cookbook learns from a fixed dataset. Reinforcement learning is
different: there's no dataset at all - the agent generates its own training data by acting in
an environment and learning from the rewards it gets back.

We implement **REINFORCE** (the simplest policy-gradient algorithm) on a tiny hand-rolled
grid world - no `gymnasium` dependency, just a 5x5 grid, a start corner, and a goal corner.
The optimal solution is exactly known (Manhattan distance), so success is trivial to measure.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

GRID = 5
START = (0, 0)
GOAL = (4, 4)
MAX_STEPS = 30
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right


def step(pos, action):
    dx, dy = ACTIONS[action]
    x, y = pos
    new_pos = (max(0, min(GRID - 1, x + dx)), max(0, min(GRID - 1, y + dy)))
    done = new_pos == GOAL
    reward = 10.0 if done else -1.0  # -1 per step pushes the agent toward the shortest path
    return new_pos, reward, done


def state_tensor(pos):
    x, y = pos
    return torch.tensor([x / (GRID - 1), y / (GRID - 1)], dtype=torch.float32, device=device)

## The policy and REINFORCE

The policy network maps a state to action probabilities. To train it:

1. Play a full episode, sampling actions from the current policy, recording each action's
   log-probability and the reward received.
2. Compute the discounted return $G_t = \sum_{k \ge t} \gamma^{k-t} r_k$ for every step -
   "how good was everything that happened from this point on."
3. Push up the probability of actions that led to high returns, push down actions that led to
   low returns: loss $= -\sum_t \log \pi(a_t \mid s_t) \cdot G_t$.

This is the core idea behind every policy-gradient method (PPO, A2C, etc. all build on it).

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 4),
        )

    def forward(self, s):
        return self.net(s)


def run_episode(policy, greedy=False):
    pos = START
    log_probs, rewards = [], []
    for _ in range(MAX_STEPS):
        logits = policy(state_tensor(pos))
        probs = torch.softmax(logits, dim=-1)
        if greedy:
            action = torch.argmax(probs).item()
        else:
            dist = torch.distributions.Categorical(probs)
            action_t = dist.sample()
            action = action_t.item()
            log_probs.append(dist.log_prob(action_t))
        pos, r, done = step(pos, action)
        rewards.append(r)
        if done:
            break
    return log_probs, rewards, pos == GOAL


def discounted_returns(rewards, gamma=0.99):
    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    return (returns - returns.mean()) / (returns.std() + 1e-8)  # variance reduction

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="/work/runs/rl")

policy = PolicyNet().to(device)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

n_episodes = 1500
recent_lens = []
for ep in range(n_episodes):
    log_probs, rewards, reached = run_episode(policy)
    returns = discounted_returns(rewards)
    loss = -torch.sum(torch.stack(log_probs) * returns)
    opt.zero_grad()
    loss.backward()
    opt.step()

    recent_lens.append(len(rewards))
    if len(recent_lens) > 100:
        recent_lens.pop(0)
    writer.add_scalar("episode/length", len(rewards), ep)
    writer.add_scalar("episode/reached_goal", float(reached), ep)
    if ep % 300 == 0:
        print(f"episode {ep:5d}  avg episode length (last 100): {np.mean(recent_lens):.1f}")

writer.close()

## Evaluate: does it find the optimal path?

With the start and goal at opposite corners of a 5x5 grid, the optimal path length (Manhattan
distance) is exactly known - no guesswork needed to check the answer.

In [ ]:
optimal_len = abs(GOAL[0] - START[0]) + abs(GOAL[1] - START[1])

lens, successes = [], 0
for _ in range(50):
    _, rewards, reached = run_episode(policy, greedy=True)
    lens.append(len(rewards))
    successes += int(reached)

print(f"optimal path length: {optimal_len}")
print(f"greedy policy: success rate {successes / 50:.2f}, average length {np.mean(lens):.1f}")

## Next steps

- Add obstacles to the grid (cells the agent can't enter) and watch the optimal path - and the
  policy's learned path - bend around them.
- Try a sparser reward (only +10 on reaching the goal, 0 otherwise, no per-step penalty) and
  see training get much slower - a classic RL exploration problem.
- Swap REINFORCE for an actor-critic method (add a value-function baseline instead of just
  normalizing returns) to reduce variance further.
- Move to `gymnasium`'s `CartPole-v1` or similar for a continuous-ish state space instead of
  this discrete grid.